In [2]:
import pandas as pd

df=pd.read_csv(r"C:\Users\skuma\OneDrive\Desktop\New folder\cleaned_data.csv")
print(df.head())
print("Shape: ",df.shape)

                                                text  sentiment
0                I`d have responded, if I were going          1
1      Sooo SAD I will miss you here in San Diego!!!          0
2                          my boss is bullying me...          0
3                     what interview! leave me alone          0
4   Sons of ****, why couldn`t they put them on t...          0
Shape:  (22900, 2)


In [3]:
#Tokennize the text
texts = df["text"].tolist()
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokens = tokenizer(
    texts,                  # list of strings
    padding=True,           # pad shorter sentences
    truncation=True,        # cut longer sentences
    max_length=128,         # max length of each text
    return_tensors="np"     # return PyTorch tensors
)
print(tokens)


C:\Users\skuma\anaconda3\envs\tf-gpu\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'input_ids': array([[  101,  1045,  1036, ...,     0,     0,     0],
       [  101, 17111,  2080, ...,     0,     0,     0],
       [  101,  2026,  5795, ...,     0,     0,     0],
       ...,
       [  101,  1045,  1036, ...,     0,     0,     0],
       [  101,  8038,  2100, ...,     0,     0,     0],
       [  101,  2035,  2023, ...,     0,     0,     0]]), 'attention_mask': array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])}


In [4]:
import numpy as np
data_labels=tokens["input_ids"]
data_target=df["sentiment"].to_numpy()
val_labels=data_labels[:2000]
val_targets=data_target[:2000]
train_labels=data_labels[2000:12000]
train_targets=data_target[2000:12000]
print("train_label_shape: ",train_labels.shape)
print("train_target_shape: ",train_targets.shape)
print("val_shape: ",val_labels.shape)
print("val_shape: ",val_targets.shape)


train_label_shape:  (10000, 95)
train_target_shape:  (10000,)
val_shape:  (2000, 95)
val_shape:  (2000,)


In [5]:
from transformers import TFDistilBertForSequenceClassification
import tensorflow as tf

model = TFDistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


C:\Users\skuma\anaconda3\envs\tf-gpu\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model 

In [6]:
with tf.device('/GPU:0'):
    history = model.fit(
        train_labels,
        train_targets,
        validation_data=(val_labels,val_targets),
        batch_size=16,
        epochs=5
        )


Epoch 1/5
625/625 [==============================] - 193s 293ms/step - loss: 0.3838 - accuracy: 0.8262 - val_loss: 0.2953 - val_accuracy: 0.8725
Epoch 2/5
625/625 [==============================] - 181s 290ms/step - loss: 0.2343 - accuracy: 0.9060 - val_loss: 0.2841 - val_accuracy: 0.8840
Epoch 3/5
625/625 [==============================] - 181s 290ms/step - loss: 0.1449 - accuracy: 0.9456 - val_loss: 0.3660 - val_accuracy: 0.8705
Epoch 4/5
625/625 [==============================] - 180s 288ms/step - loss: 0.0798 - accuracy: 0.9734 - val_loss: 0.4055 - val_accuracy: 0.8605
Epoch 5/5
625/625 [==============================] - 181s 290ms/step - loss: 0.0548 - accuracy: 0.9839 - val_loss: 0.5107 - val_accuracy: 0.8610


In [10]:
model.save("sentiment_review_model")


INFO:tensorflow:Assets written to: sentiment_review_model\assets


INFO:tensorflow:Assets written to: sentiment_review_model\assets


In [23]:
df2=pd.read_csv(r"C:\Users\skuma\OneDrive\Desktop\New folder\test.csv",encoding="latin1")
print("Shape: ",df2.shape)
stmt={"neutral":1,"positive":1,"negative":0}
test_label_data=df2["text"]
df2 = df2.dropna(subset=["text"])
df2["sentiment"] = df2["sentiment"].map(stmt)
y_test=df2["sentiment"].to_numpy()
df["text"]=df["text"].astype(str)
text2=df2["text"].to_list()
token2 = tokenizer(
    text2,                  # list of strings
    padding=True,           # pad shorter sentences
    truncation=True,        # cut longer sentences
    max_length=128,         # max length of each text
    return_tensors="np"     
)
x_test=token2["input_ids"]
x_test=x_test[:1000]
y_test=y_test[:1000]
loss, acc = model.evaluate(x_test, y_test)
print(acc)

Shape:  (4815, 9)
32/32 [==============================] - 5s 135ms/step - loss: 0.4798 - accuracy: 0.8630
0.8629999756813049


In [38]:
import tensorflow as tf
text = "I am sick with the hotel condition"
token3 = tokenizer(
    [text],
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="tf"
)

X_predict = {
    "input_ids": token3["input_ids"],
    "attention_mask": token3["attention_mask"]
}

outputs = model.predict(X_predict)

logits = outputs.logits
print("Logits:", logits)

probs = tf.nn.softmax(logits, axis=1)
print("Probabilities:", probs.numpy())

pred_class = tf.argmax(probs, axis=1).numpy()[0]

if pred_class == 0:
    print("Review is Negative ❌")
else:
    print("Review is Positive ✅")


1/1 [==============================] - 0s 90ms/step
Logits: [[ 2.9687593 -3.0170727]]
Probabilities: [[0.9974922  0.00250782]]
Review is Negative ❌
